# M2 Validation: Pi0.5 (OpenPI) VLA Service (Linux/Colab)

End-to-end M2 acceptance on a Google Colab Linux+CUDA VM (run from Cursor IDE, no `google.colab` imports).
Covers environment setup, Pi0.5 (OpenPI) VLA server startup, POST /predict sanity check, and the project test suite.

## 1 — Secrets & clone

In [ ]:
import getpass, os, subprocess, sys

os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT (repo scope): ')

subprocess.run(
    ['git', 'clone', f"https://{os.environ['GITHUB_TOKEN']}@github.com/pmish11/Longor-Robotics.git", '/content/Longor'],
    check=True
)
os.chdir('/content/Longor')
sys.path.insert(0, '/content/Longor/src')

## 2 — Install app deps

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)

pip('install', '-U', 'pip')
pip('install', '-r', 'requirements.txt')
pip('install', 'Pillow')  # POST /predict sanity check (section 7)

## 3 — Install Pi0.5 VLA venv

In [ ]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    result = subprocess.run(
        [VLA_PYTHON, '-m', 'pip', *args],
        capture_output=True, text=True
    )
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args)

# Colab (Debian/Ubuntu) doesn't ship python3-venv; virtualenv works without OS packages.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'virtualenv'], check=True)
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
else:
    print('.vla-venv already exists, skipping creation.')
vla_pip('install', '-U', 'pip', 'setuptools', 'wheel')
vla_pip('install', '-U', 'jax[cuda12]')
if not os.path.exists('/root/openpi'):
    subprocess.run(
        ['git', 'clone', '--recurse-submodules', 'https://github.com/Physical-Intelligence/openpi.git', '/root/openpi'],
        check=True
    )
else:
    print('/root/openpi already exists, skipping clone.')
vla_pip('install', '-e', '/root/openpi')
# OpenPI pins a specific lerobot commit (pyproject.toml rev); PyPI lerobot has a different package structure
vla_pip('install', 'git+https://github.com/huggingface/lerobot@0cf864870cf29f4738d3ade893e6fd13fbd7cdb5')
vla_pip('install', '-r', '/content/Longor/services/requirements-vla-pi05-linux.txt')

## 4 — Write `.env`

In [ ]:
import os

overrides = {
    'VLA_POLICY': 'pi05',
    'VLA_PORT': '8765',
    'OPENPI_CONFIG': 'pi05_droid',
    'BACKEND': 'maniskill',
}

lines_out = []
seen_keys = set()
with open('/content/Longor/.env.example') as f:
    for line in f:
        stripped = line.rstrip('\n\r')
        if stripped and not stripped.startswith('#') and '=' in stripped:
            key, _, _ = stripped.partition('=')
            key = key.strip()
            if key in overrides:
                lines_out.append(f'{key}={overrides[key]}')
                seen_keys.add(key)
                continue
        lines_out.append(stripped)

for key, val in overrides.items():
    if key not in seen_keys:
        lines_out.append(f'{key}={val}')

with open('/content/Longor/.env', 'w') as f:
    f.write('\n'.join(lines_out) + '\n')
print('Wrote /content/Longor/.env')

## 5 — Start VLA server

In [ ]:
import subprocess, time, urllib.request, json

VLA_LOG = '/tmp/vla_pi05_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'

def _tail_log(n=40):
    try:
        with open(VLA_LOG) as f:
            lines = f.readlines()
        return ''.join(lines[-n:])
    except FileNotFoundError:
        return '(log file not created yet)'

# Pi0.5 blocks /health until checkpoint download + model load (GCS on first run can be slow).
log = open(VLA_LOG, 'w', buffering=1)
vla_proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    stdout=log, stderr=subprocess.STDOUT, cwd='/content/Longor'
)

POLL_SEC = 15
MAX_WAIT_SEC = 20 * 60
for i in range(MAX_WAIT_SEC // POLL_SEC):
    time.sleep(POLL_SEC)
    elapsed = (i + 1) * POLL_SEC

    if vla_proc.poll() is not None:
        print(_tail_log(60))
        raise RuntimeError(f'VLA server exited early (code {vla_proc.returncode}). Log above.')

    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('VLA server ready:', json.loads(resp.read()))
        break
    except Exception:
        print(f'Waiting... ({elapsed}s)')
        if elapsed % 60 == 0:
            print('--- recent log ---')
            print(_tail_log(15))
else:
    print(_tail_log(60))
    raise RuntimeError(f'VLA server did not come up in {MAX_WAIT_SEC // 60} minutes. Log above.')

## 6 — VLA server log (diagnostic)

In [ ]:
with open('/tmp/vla_pi05_server.log') as f:
    lines = f.readlines()
print(''.join(lines[-30:]))

## 7 — POST /predict sanity check

In [ ]:
import io, json, urllib.request
from PIL import Image
import numpy as np

# synthetic white 224x224 image
img = Image.fromarray(np.ones((224, 224, 3), dtype=np.uint8) * 128)
buf = io.BytesIO()
img.save(buf, format='JPEG')
jpeg_bytes = buf.getvalue()

boundary = b'----FormBoundary'
body = (
    b'--' + boundary + b'\r\n'
    b'Content-Disposition: form-data; name="image"; filename="test.jpg"\r\n'
    b'Content-Type: image/jpeg\r\n\r\n' + jpeg_bytes + b'\r\n'
    b'--' + boundary + b'\r\n'
    b'Content-Disposition: form-data; name="task"\r\n\r\n'
    b'pick up the cube\r\n'
    b'--' + boundary + b'--\r\n'
)
req = urllib.request.Request(
    'http://localhost:8765/predict',
    data=body,
    headers={'Content-Type': f'multipart/form-data; boundary={boundary.decode()}'},
    method='POST',
)
resp = urllib.request.urlopen(req, timeout=180)
result = json.loads(resp.read())
actions = result['actions']
assert isinstance(actions, list) and len(actions) > 0
assert all(len(a) == 4 for a in actions), f'Expected [dx,dy,dz,gripper] per step, got: {actions[0]}'
print(f'POST /predict OK: {len(actions)} action steps, first={actions[0]}')

## 8 — Run pytest

In [ ]:
import os, subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-m', 'not integration', '--tb=short', '-q'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PYTHONPATH': '/content/Longor/src'}
)
print(result.stdout[-4000:])
print(result.stderr[-1000:])
assert result.returncode == 0, f'pytest failed (exit {result.returncode})'

## 9 — Teardown

In [ ]:
vla_proc.terminate()
print('VLA server stopped.')